In [1]:
!pip install transformers sentencepiece scikit-learn -q

In [2]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

drive_path = "/content/drive/MyDrive/Event Log/2025-7-28 申毅实习/策略研究/NLP 日度"
os.chdir(drive_path)
print("Working directory:", os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/MyDrive/Event Log/2025-7-28 申毅实习/策略研究/NLP 日度


In [3]:
import torch
import warnings
import logging
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification
)
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger().setLevel(logging.ERROR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(device)}")

Using GPU: NVIDIA A100-SXM4-40GB


In [4]:
device = 0 if torch.cuda.is_available() else -1
print("Device set to", "GPU" if device == 0 else "CPU")

Device set to GPU


In [5]:
# === Translator: facebook/nllb-200-3.3B ===
translator_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-3.3B")
translator_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-3.3B").to(device)

# === Classifier: FinBERT-Tone ===
classifier_tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
classifier_model = AutoModelForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone").to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [6]:
equity_reports = pd.read_csv(
    'equity_reports.csv',
    encoding='utf-8',
    parse_dates=['PUBLISH_DATE', 'UPDATE_TIME', 'UPDATE_DATE']
)

In [7]:
output_path = "test.csv"

if os.path.exists(output_path):
    processed_ids = set(pd.read_csv(output_path)["REPORT_ID"])
    print(f"Resuming from progress log: {len(processed_ids)}")
else:
    processed_ids = set()
    pd.DataFrame(columns=[
        "REPORT_ID", "title_label", "title_score", "summary_label", "summary_score"
    ]).to_csv(output_path, index=False)
    print("Created new progress log.")

Resuming from progress log: 0


# Batch Loader

In [11]:
class ReportDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe[~dataframe["REPORT_ID"].isin(processed_ids)].copy()
        self.df.fillna("", inplace=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "REPORT_ID": row["REPORT_ID"],
            "TITLE": row["TITLE"],
            "SUMMARY": row["SUMMARY"]
        }

batch_size = 5
dataset = ReportDataset(equity_reports)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

In [12]:
def translate_batch(texts, max_length):
    inputs = translator_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        outputs = translator_model.generate(
            **inputs,
            max_length=max_length,
            num_beams=4,
            early_stopping=True
        )

    return translator_tokenizer.batch_decode(outputs, skip_special_tokens=True)

def classify_batch(texts):
    inputs = classifier_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        logits = classifier_model(**inputs).logits
        probs = torch.nn.functional.softmax(logits, dim=-1)
        scores, preds = torch.max(probs, dim=1)
        labels = [classifier_model.config.id2label[p.item()] for p in preds]

    return labels, scores.tolist()

In [13]:
for batch_idx, batch in enumerate(tqdm(dataloader, desc="Processing First 12 Batches")):
    if batch_idx >= 12:
        break

    ids = batch["REPORT_ID"]
    titles_cn = batch["TITLE"]
    summaries_cn = batch["SUMMARY"]

    titles_en = translate_batch(titles_cn, max_length=600)
    summaries_en = translate_batch(summaries_cn, max_length=1024)

    title_labels, title_scores = classify_batch(titles_en)
    summary_labels, summary_scores = classify_batch(summaries_en)

    df_batch = pd.DataFrame({
        "REPORT_ID": ids,
        "title_label": title_labels,
        "title_score": title_scores,
        "summary_label": summary_labels,
        "summary_score": summary_scores
    })

    df_batch.to_csv(output_path, mode="a", header=False, index=False)
    print(f"[Batch {batch_idx + 1}] Saved {len(ids)} rows to {output_path}")


Processing First 12 Batches:   0%|          | 1/64891 [00:20<376:07:18, 20.87s/it]

[Batch 1] Saved 5 rows to test.csv


Processing First 12 Batches:   0%|          | 2/64891 [00:41<372:18:18, 20.66s/it]

[Batch 2] Saved 5 rows to test.csv


Processing First 12 Batches:   0%|          | 2/64891 [01:17<697:50:04, 38.72s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 88.00 MiB. GPU 0 has a total capacity of 39.56 GiB of which 46.88 MiB is free. Process 47834 has 39.50 GiB memory in use. Of the allocated memory 36.66 GiB is allocated by PyTorch, and 2.35 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)